In [6]:
import os
import gc
import copy
import datetime
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from netCDF4 import Dataset as ncDataset
import rasterio
from tqdm import tqdm
import optuna
import logging
import json
from torch.utils.tensorboard import SummaryWriter

# Set up logging
log_dir = "rainfall_gan_logs"
if not os.path.exists(log_dir):
    os.makedirs(log_dir)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(log_dir, 'training.log')),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger('RainfallGAN')

# Dataset definition
class RainfallDataset(Dataset):
    def __init__(self, low_res_folder, high_res_folder, dem_file, start_date, end_date):
        self.low_res_folder = low_res_folder
        self.high_res_folder = high_res_folder
        self.dem_file = dem_file
        self.dates = [start_date + datetime.timedelta(days=i) for i in range((end_date - start_date).days + 1)]
        self.valid_data = self._filter_valid_data()
        if len(self.valid_data) > 0:
            self.dem_data = self._load_dem_data()
        else:
            self.dem_data = None
            logger.warning("유효한 데이터 포인트가 없습니다. DEM 데이터를 로드하지 않았습니다.")
    
    def __len__(self):
        return len(self.valid_data)
    
    def __getitem__(self, idx):
        try:
            date = self.valid_data[idx]
            low_res_file = os.path.join(self.low_res_folder, date.strftime("%Y%m%d") + ".nc4")
            # 여기서 파일 이름 패턴 수정: %Y.%m.%d -> %Y%m%d
            high_res_file = os.path.join(self.high_res_folder, date.strftime("%Y%m%d") + "_Cokriging_Precipitation.nc")
            
            logger.debug(f"저해상도 파일 로드 중: {low_res_file}")
            logger.debug(f"고해상도 파일 로드 중: {high_res_file}")
            
            low_res_data = self.load_and_resize_nc_file(low_res_file, 'precipitation', target_shape=(61, 101))
            high_res_data = self.load_nc_file(high_res_file, 'precipitation')
            dem_data = self.dem_data
            
            if low_res_data is None or high_res_data is None:
                raise FileNotFoundError(f"날짜 {date.strftime('%Y-%m-%d')}에 대한 데이터가 없습니다")
            
            # Normalize data
            low_res_data = self.normalize_data(low_res_data)
            dem_data = self.normalize_data(dem_data)
            
            # Add month as sine and cosine
            month = date.month
            month_sin = np.sin(2 * np.pi * month / 12)
            month_cos = np.cos(2 * np.pi * month / 12)
            
            # Add month information to channels
            month_sin_map = np.full(low_res_data.shape, month_sin, dtype=np.float32)
            month_cos_map = np.full(low_res_data.shape, month_cos, dtype=np.float32)
            
            low_res_data = torch.tensor(low_res_data, dtype=torch.float32).unsqueeze(0)
            high_res_data = torch.tensor(high_res_data, dtype=torch.float32).unsqueeze(0)
            dem_data = torch.tensor(dem_data, dtype=torch.float32).unsqueeze(0)
            month_sin_map = torch.tensor(month_sin_map, dtype=torch.float32).unsqueeze(0)
            month_cos_map = torch.tensor(month_cos_map, dtype=torch.float32).unsqueeze(0)
            
            # Combine channels: low_res, dem, sin(month), cos(month)
            input_data = torch.cat([low_res_data, dem_data, month_sin_map, month_cos_map], dim=0)
            
            return input_data, high_res_data
        except Exception as e:
            logger.error(f"인덱스 {idx}에 대한 __getitem__에서 오류 발생: {str(e)}")
            raise e
    
    def normalize_data(self, data):
        mean = np.mean(data)
        std = np.std(data)
        if std == 0:
            return data - mean
        return (data - mean) / (std + 1e-8)
    
    def load_nc_file(self, file_path, var_name):
        try:
            with ncDataset(file_path, 'r') as nc_file:
                data = nc_file.variables[var_name][:].astype(np.float32)
                return data
        except FileNotFoundError:
            logger.warning(f"파일 {file_path}을(를) 찾을 수 없습니다.")
            return None
        except Exception as e:
            logger.error(f"{file_path} 로드 중 오류 발생: {str(e)}")
            return None
    
    def load_and_resize_nc_file(self, file_path, var_name, target_shape):
        try:
            with ncDataset(file_path, 'r') as nc_file:
                data = nc_file.variables[var_name][:].astype(np.float32)
                if len(data.shape) == 3:
                    data = data.squeeze(0)
                data = torch.tensor(data, dtype=torch.float32)
                data = torch.nn.functional.interpolate(
                    data.unsqueeze(0).unsqueeze(0), 
                    size=target_shape, 
                    mode='bilinear', 
                    align_corners=False
                ).squeeze()
                return data.numpy()
        except FileNotFoundError:
            logger.warning(f"파일 {file_path}을(를) 찾을 수 없습니다.")
            return None
        except Exception as e:
            logger.error(f"{file_path} 리사이징 중 오류 발생: {str(e)}")
            return None
    
    def _filter_valid_data(self):
        valid_dates = []
        # 처음 몇 개의 파일에 대해 디버그 출력 추가
        for i, date in enumerate(self.dates):
            low_res_file = os.path.join(self.low_res_folder, date.strftime("%Y%m%d") + ".nc4")
            # 여기서 파일 이름 패턴 수정: %Y.%m.%d -> %Y%m%d
            high_res_file = os.path.join(self.high_res_folder, date.strftime("%Y%m%d") + "_Cokriging_Precipitation.nc")
            
            # 처음 5개 파일에 대해 디버그 정보 출력
            if i < 5:
                logger.info(f"저해상도 확인: {low_res_file}, 존재: {os.path.exists(low_res_file)}")
                logger.info(f"고해상도 확인: {high_res_file}, 존재: {os.path.exists(high_res_file)}")
            
            if os.path.exists(low_res_file) and os.path.exists(high_res_file):
                valid_dates.append(date)
        
        logger.info(f"전체 {len(self.dates)}개 날짜 중 {len(valid_dates)}개의 유효한 데이터 포인트를 찾았습니다.")
        
        # 유효한 날짜가 없는 경우 디렉터리의 파일 몇 개를 확인하여 디버깅에 도움이 되도록 함
        if len(valid_dates) == 0:
            try:
                low_res_files = os.listdir(self.low_res_folder)[:5]
                high_res_files = os.listdir(self.high_res_folder)[:5]
                logger.warning(f"유효한 쌍을 찾을 수 없습니다. 저해상도 파일 샘플: {low_res_files}")
                logger.warning(f"고해상도 파일 샘플: {high_res_files}")
            except Exception as e:
                logger.error(f"디렉터리 내용 나열 중 오류 발생: {str(e)}")
        
        return valid_dates
    
    def _load_dem_data(self):
        try:
            with rasterio.open(self.dem_file) as dem_src:
                dem_data = dem_src.read(1)
                dem_data = self.normalize_data(dem_data)
                dem_data = torch.tensor(dem_data, dtype=torch.float32)
                dem_data = torch.nn.functional.interpolate(
                    dem_data.unsqueeze(0).unsqueeze(0), 
                    size=(61, 101), 
                    mode='bilinear', 
                    align_corners=False
                ).squeeze()
                return dem_data.numpy()
        except FileNotFoundError:
            logger.error(f"DEM 파일 {self.dem_file}을(를) 찾을 수 없습니다.")
            return None
        except Exception as e:
            logger.error(f"DEM 파일 {self.dem_file} 로드 중 오류 발생: {str(e)}")
            return None

# 파일 이름 패턴 확인 함수
def check_file_patterns(low_res_folder, high_res_folder):
    """
    데이터 폴더의 파일 이름 패턴을 확인하는 헬퍼 함수
    """
    logger.info("파일 이름 패턴 분석 중...")
    
    # 저해상도 파일 확인
    try:
        low_res_files = os.listdir(low_res_folder)[:10]  # 처음 10개 파일 가져오기
        if low_res_files:
            logger.info(f"저해상도 파일 예시: {low_res_files}")
    except Exception as e:
        logger.error(f"저해상도 폴더 읽기 오류: {str(e)}")
    
    # 고해상도 파일 확인
    try:
        high_res_files = os.listdir(high_res_folder)[:10]  # 처음 10개 파일 가져오기
        if high_res_files:
            logger.info(f"고해상도 파일 예시: {high_res_files}")
    except Exception as e:
        logger.error(f"고해상도 폴더 읽기 오류: {str(e)}")
    
    # 패턴 유추하기
    low_res_pattern = None
    high_res_pattern = None
    
    if len(low_res_files) > 0:
        sample = low_res_files[0]
        if sample.endswith('.nc4') and len(sample) >= 12:
            date_part = sample[:8]  # YYYYMMDD 형식 가정
            if date_part.isdigit():
                low_res_pattern = f"YYYYMMDD.nc4"
    
    if len(high_res_files) > 0:
        sample = high_res_files[0]
        if '_Cokriging_Precipitation.nc' in sample:
            date_part = sample.split('_')[0]  # 언더스코어 이전의 첫 부분 가져오기
            if len(date_part) >= 8 and date_part.isdigit():
                high_res_pattern = f"YYYYMMDD_Cokriging_Precipitation.nc"
            elif '.' in date_part and len(date_part.replace('.', '')) >= 8:
                high_res_pattern = f"YYYY.MM.DD_Cokriging_Precipitation.nc"
    
    logger.info(f"감지된 저해상도 패턴: {low_res_pattern}")
    logger.info(f"감지된 고해상도 패턴: {high_res_pattern}")
    
    return low_res_pattern, high_res_pattern

# Discriminator Model
class Discriminator(nn.Module):
    def __init__(self, dropout_rate=0.3):
        super(Discriminator, self).__init__()
        
        self.main = nn.Sequential(
            # Layer 1
            nn.Conv2d(2, 64, kernel_size=3, stride=2, padding=1),  # 4x4 -> 3x3 커널로 변경
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(dropout_rate),
            
            # Layer 2
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),  # 4x4 -> 3x3 커널로 변경
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(dropout_rate),
            
            # Layer 3
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),  # 4x4 -> 3x3 커널로 변경
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(dropout_rate),
            
            # Layer 4
            nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1),  # 4x4 -> 3x3 커널로 변경
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(dropout_rate),
            
            # Layer 5
            nn.Conv2d(512, 1, kernel_size=3, stride=1, padding=0),  # 4x4 -> 3x3 커널로 변경
            nn.Sigmoid()
        )
        
    def forward(self, high_res, low_res):
        x = torch.cat([high_res, low_res[:, :1, :, :]], dim=1)  # Only use the first channel of low_res
        return self.main(x).view(-1, 1)


# Generator Model
class Generator(nn.Module):
    def __init__(self, dropout_rate=0.3, attention_channels=256):
        super(Generator, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(4, 64, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, attention_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(attention_channels),
            nn.LeakyReLU(0.2)
        )
        
        # Enhanced attention module
        self.attention = nn.Sequential(
            nn.Conv2d(attention_channels, attention_channels, kernel_size=1),
            nn.BatchNorm2d(attention_channels),
            nn.ReLU(),
            nn.Conv2d(attention_channels, attention_channels, kernel_size=1),
            nn.Sigmoid()
        )
        
        # Rainfall amplification branch
        self.rainfall_amp = nn.Sequential(
            nn.Conv2d(attention_channels, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 1, kernel_size=1),
            nn.ReLU()
        )
        
        self.decoder = nn.Sequential(
            nn.Conv2d(attention_channels, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Dropout2d(dropout_rate),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Dropout2d(dropout_rate),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            nn.Conv2d(32, 1, kernel_size=3, padding=1),
            nn.ReLU()
        )
        
        # Enhanced rainfall intensity layer with skip connections
        self.enhancement = nn.ModuleList([
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.Conv2d(32, 1, kernel_size=1)
        ])
        
    def forward(self, x, target_shape):
        # Encoder
        enc = self.encoder(x)
        
        # Enhanced attention
        att = self.attention(enc)
        enc_att = enc * att
        
        # Rainfall amplification branch
        amp = self.rainfall_amp(enc_att)
        
        # Decoder
        dec = self.decoder(enc_att)
        
        # Enhancement with skip connections
        enhanced = dec
        for i, layer in enumerate(self.enhancement):
            if i == len(self.enhancement) - 1:
                enhanced = layer(enhanced) + amp  # Add amplification
            else:
                enhanced = F.relu(layer(enhanced))
        
        # Ensure non-negative values and apply slight amplification
        enhanced = F.relu(enhanced) * 1.2  # Amplification factor
        
        # Interpolate to target shape
        output = torch.nn.functional.interpolate(
            enhanced, 
            size=target_shape, 
            mode='bilinear', 
            align_corners=False
        )
        
        return output

# Modified loss function with stronger underestimation penalty
class ElevationAwareLoss(nn.Module):
    def __init__(self, dem_data, alpha=0.4, beta=0.4):
        super(ElevationAwareLoss, self).__init__()
        self.dem_data = torch.tensor(dem_data).cuda()
        self.alpha = alpha
        self.beta = beta
        
    def forward(self, pred, target):
        # Basic MSE loss
        mse_loss = torch.mean((pred - target) ** 2)
        
        # Elevation-weighted loss
        elevation_weight = torch.exp(self.dem_data / self.dem_data.max())
        weighted_loss = torch.mean(elevation_weight * (pred - target) ** 2)
        
        # Enhanced rainfall intensity penalty
        intensity_diff = target - pred
        underestimation_mask = (intensity_diff > 0).float()
        intensity_threshold = target.mean() * 0.5  # Threshold for heavy rainfall
        heavy_rainfall_mask = (target > intensity_threshold).float()
        
        # Stronger penalty for underestimating heavy rainfall
        intensity_loss = torch.mean(
            underestimation_mask * heavy_rainfall_mask * (intensity_diff ** 2) * 4.0 +  # Very strong penalty for heavy rainfall underestimation
            underestimation_mask * (1 - heavy_rainfall_mask) * (intensity_diff ** 2) * 2.5 +  # Strong penalty for normal rainfall underestimation
            (1 - underestimation_mask) * (intensity_diff ** 2)  # Normal penalty for overestimation
        )
        
        return (1 - self.alpha - self.beta) * mse_loss + self.alpha * weighted_loss + self.beta * intensity_loss

# RainfallGAN class with modified training strategy
class RainfallGAN:
    def __init__(self, dem_data, config):
        self.config = config
        
        # Initialize models with hyperparameters
        self.generator = Generator(
            dropout_rate=config['generator_dropout'],
            attention_channels=config['attention_channels']
        ).cuda()
        
        self.discriminator = Discriminator(
            dropout_rate=config['discriminator_dropout']
        ).cuda()
        
        self.dem_data = torch.tensor(dem_data).cuda()
        
        # Optimizers with configurable learning rates
        self.g_optimizer = optim.Adam(
            self.generator.parameters(), 
            lr=config['g_learning_rate'], 
            betas=(config['g_beta1'], config['g_beta2'])
        )
        
        self.d_optimizer = optim.Adam(
            self.discriminator.parameters(), 
            lr=config['d_learning_rate'], 
            betas=(config['d_beta1'], config['d_beta2'])
        )
        
        # Learning rate schedulers
        self.g_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.g_optimizer, 
            mode='min',
            factor=0.5,
            patience=5,
            verbose=True
        )
        
        self.d_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.d_optimizer, 
            mode='min',
            factor=0.5,
            patience=5,
            verbose=True
        )
        
        self.criterion_gan = nn.BCELoss()
        self.criterion_l1 = nn.L1Loss()
        self.criterion_elevation = ElevationAwareLoss(
            dem_data, 
            alpha=config['elevation_alpha'], 
            beta=config['elevation_beta']
        )
    
    def train_step(self, low_res, high_res):
        batch_size = low_res.size(0)
        real_label = torch.ones(batch_size, 1).cuda()
        fake_label = torch.zeros(batch_size, 1).cuda()

        # Train Discriminator
        self.d_optimizer.zero_grad()
        d_real = self.discriminator(high_res, low_res)
        real_labels = torch.ones_like(d_real)            # Discriminator 출력(shape)에 맞춰 label 생성
        d_real_loss = self.criterion_gan(d_real, real_labels)

        fake_high_res = self.generator(low_res, (61, 101))
        d_fake = self.discriminator(fake_high_res.detach(), low_res)
        fake_labels = torch.zeros_like(d_fake)           # 마찬가지로 zeros_like
        d_fake_loss = self.criterion_gan(d_fake, fake_labels)

        d_loss = (d_real_loss + d_fake_loss) * 0.5
        d_loss.backward()
        self.d_optimizer.step()

        # Train Generator
        self.g_optimizer.zero_grad()
        g_fake = self.discriminator(fake_high_res, low_res)
        gen_real_labels = torch.ones_like(g_fake)        # Generator 쪽도 real label로
        g_gan_loss = self.criterion_gan(g_fake, gen_real_labels)

        # Content & Elevation & Intensity losses (unchanged)
        g_content_loss = self.criterion_l1(fake_high_res, high_res) * self.config['content_weight']
        g_elevation_loss = self.criterion_elevation(fake_high_res, high_res) * self.config['elevation_weight']
        intensity_diff = torch.mean(torch.relu(high_res - fake_high_res))
        g_intensity_loss = intensity_diff * self.config['intensity_weight']

        # Total generator loss
        g_loss = g_gan_loss + g_content_loss + g_elevation_loss + g_intensity_loss
        g_loss.backward()
        self.g_optimizer.step()

        return {
            'd_loss': d_loss.item(),
            'g_loss': g_loss.item(),
            'g_gan_loss': g_gan_loss.item(),
            'g_content_loss': g_content_loss.item(),
            'g_elevation_loss': g_elevation_loss.item(),
            'g_intensity_loss': g_intensity_loss.item()
        }

# Early stopping implementation
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0, mode='min'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False
    
    def __call__(self, val_loss, model=None):
        score = -val_loss if self.mode == 'min' else val_loss
        
        if self.best_score is None:
            self.best_score = score
            return False
        
        delta = score - self.best_score
        if self.mode == 'min':
            delta = -delta
            
        if delta < self.min_delta:
            self.counter += 1
            logger.info(f'EarlyStopping 카운터: {self.counter} / {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
                return True
        else:
            self.best_score = score
            self.counter = 0
            return False

# Hyperparameter optimization with Optuna
def objective(trial, train_data, val_data, dem_data, epochs=20):
    # Configure hyperparameters for this trial
    config = {
        'g_learning_rate': trial.suggest_float('g_learning_rate', 1e-5, 1e-3, log=True),
        'd_learning_rate': trial.suggest_float('d_learning_rate', 1e-5, 1e-3, log=True),
        'g_beta1': trial.suggest_float('g_beta1', 0.3, 0.9),
        'g_beta2': trial.suggest_float('g_beta2', 0.8, 0.999),
        'd_beta1': trial.suggest_float('d_beta1', 0.3, 0.9),
        'd_beta2': trial.suggest_float('d_beta2', 0.8, 0.999),
        'content_weight': trial.suggest_float('content_weight', 50.0, 200.0),
        'elevation_weight': trial.suggest_float('elevation_weight', 5.0, 50.0),
        'intensity_weight': trial.suggest_float('intensity_weight', 10.0, 100.0),
        'generator_dropout': trial.suggest_float('generator_dropout', 0.1, 0.5),
        'discriminator_dropout': trial.suggest_float('discriminator_dropout', 0.1, 0.5),
        'attention_channels': trial.suggest_categorical('attention_channels', [128, 256, 384, 512]),
        'elevation_alpha': trial.suggest_float('elevation_alpha', 0.2, 0.6),
        'elevation_beta': trial.suggest_float('elevation_beta', 0.2, 0.6),
        'batch_size': trial.suggest_categorical('batch_size', [8, 16, 32]),
        'n_critic': trial.suggest_int('n_critic', 1, 5)
    }
    
    # Sample a smaller subset for faster optimization
    train_subset_size = min(len(train_data), 500)  # Limit to 500 samples for faster trials
    val_subset_size = min(len(val_data), 100)
    
    # Create subset samplers
    train_subset_indices = np.random.choice(len(train_data), train_subset_size, replace=False)
    val_subset_indices = np.random.choice(len(val_data), val_subset_size, replace=False)
    
    # Create data loaders with the subsets
    train_loader = DataLoader(
        train_data, 
        batch_size=config['batch_size'],
        sampler=torch.utils.data.SubsetRandomSampler(train_subset_indices),
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_data, 
        batch_size=config['batch_size'],
        sampler=torch.utils.data.SubsetRandomSampler(val_subset_indices),
        shuffle=False, 
        pin_memory=True
    )
    
    # Initialize GAN with this trial's hyperparameters
    gan_model = RainfallGAN(dem_data, config)
    
    # TensorBoard writer for this trial
    writer = SummaryWriter(f"logs/trial_{trial.number}")
    
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        gan_model.generator.train()
        gan_model.discriminator.train()
        
        total_g_loss = 0
        total_d_loss = 0
        total_batches = 0
        
        for i, (low_res, high_res) in enumerate(train_loader):
            low_res, high_res = low_res.cuda(), high_res.cuda()
            
            if i % config['n_critic'] == 0:
                losses = gan_model.train_step(low_res, high_res)
                total_g_loss += losses['g_loss']
            
            total_d_loss += losses['d_loss']
            total_batches += 1
            
            # Report intermediate values for pruning
            if i % 10 == 0:
                trial.report(losses['g_loss'], epoch * len(train_loader) + i)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()
        
        avg_g_loss = total_g_loss / total_batches
        avg_d_loss = total_d_loss / total_batches
        
        # Validation
        gan_model.generator.eval()
        val_loss = 0
        
        with torch.no_grad():
            for low_res, high_res in val_loader:
                low_res, high_res = low_res.cuda(), high_res.cuda()
                fake_high_res = gan_model.generator(low_res, (61, 101))
                val_loss += gan_model.criterion_l1(fake_high_res, high_res).item()
        
        avg_val_loss = val_loss / len(val_loader)
        
        # Log metrics to TensorBoard
        writer.add_scalar('Loss/train_generator', avg_g_loss, epoch)
        writer.add_scalar('Loss/train_discriminator', avg_d_loss, epoch)
        writer.add_scalar('Loss/validation', avg_val_loss, epoch)
        
        logger.info(f"Trial {trial.number}, Epoch [{epoch+1}/{epochs}], G Loss: {avg_g_loss:.4f}, D Loss: {avg_d_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
        
        # Update learning rate schedulers
        gan_model.g_scheduler.step(avg_val_loss)
        gan_model.d_scheduler.step(avg_val_loss)
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
        
        # Clean up GPU memory
        torch.cuda.empty_cache()
        gc.collect()
    
    writer.close()
    return best_val_loss

# Training function with early stopping
def train_gan_model(train_data, val_data, dem_data, config, epochs=150, model_dir="models"):
    if not os.path.exists(model_dir):
        os.makedirs(model_dir)
    
    # Create data loaders
    train_loader = DataLoader(train_data, batch_size=config['batch_size'], shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_data, batch_size=config['batch_size'], shuffle=False, pin_memory=True)
    
    # Initialize GAN with configured hyperparameters
    gan_model = RainfallGAN(dem_data, config)
    
    # TensorBoard for logging
    writer = SummaryWriter(log_dir=os.path.join(log_dir, "tensorboard"))
    
    # Initialize early stopping
    early_stopping = EarlyStopping(patience=config.get('early_stopping_patience', 15), min_delta=0.001)
    
    best_val_loss = float('inf')
    best_generator = None
    best_epoch = 0
    
    # Save hyperparameters
    with open(os.path.join(model_dir, 'hyperparameters.json'), 'w') as f:
        json.dump(config, f, indent=4)
    
    logger.info(f"하이퍼파라미터로 훈련 시작: {config}")
    
    for epoch in range(epochs):
        gan_model.generator.train()
        gan_model.discriminator.train()
        
        total_g_loss = 0
        total_d_loss = 0
        total_g_content_loss = 0
        total_g_elevation_loss = 0
        total_g_intensity_loss = 0
        total_batches = 0
        
        # Training loop
        for i, (low_res, high_res) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")):
            low_res, high_res = low_res.cuda(), high_res.cuda()
            
            if i % config['n_critic'] == 0:
                losses = gan_model.train_step(low_res, high_res)
                total_g_loss += losses['g_loss']
                total_g_content_loss += losses['g_content_loss']
                total_g_elevation_loss += losses['g_elevation_loss']
                total_g_intensity_loss += losses['g_intensity_loss']
            
            total_d_loss += losses['d_loss']
            total_batches += 1
        
        # Calculate average losses
        avg_g_loss = total_g_loss / total_batches
        avg_d_loss = total_d_loss / total_batches
        avg_g_content_loss = total_g_content_loss / total_batches
        avg_g_elevation_loss = total_g_elevation_loss / total_batches
        avg_g_intensity_loss = total_g_intensity_loss / total_batches
        
        # Validation
        gan_model.generator.eval()
        val_loss = 0
        val_rmse = 0
        
        with torch.no_grad():
            for low_res, high_res in val_loader:
                low_res, high_res = low_res.cuda(), high_res.cuda()
                fake_high_res = gan_model.generator(low_res, (61, 101))
                
                # L1 loss for validation monitoring
                val_loss += gan_model.criterion_l1(fake_high_res, high_res).item()
                
                # RMSE for validation
                val_rmse += torch.sqrt(torch.mean((fake_high_res - high_res) ** 2)).item()
        
        avg_val_loss = val_loss / len(val_loader)
        avg_val_rmse = val_rmse / len(val_loader)
        
        # Log metrics to TensorBoard
        writer.add_scalar('Loss/train_generator', avg_g_loss, epoch)
        writer.add_scalar('Loss/train_discriminator', avg_d_loss, epoch)
        writer.add_scalar('Loss/validation', avg_val_loss, epoch)
        writer.add_scalar('Loss/val_rmse', avg_val_rmse, epoch)
        writer.add_scalar('Loss/g_content_loss', avg_g_content_loss, epoch)
        writer.add_scalar('Loss/g_elevation_loss', avg_g_elevation_loss, epoch)
        writer.add_scalar('Loss/g_intensity_loss', avg_g_intensity_loss, epoch)
        
        # Log all metrics to file
        logger.info(f"Epoch [{epoch+1}/{epochs}], "
                    f"G Loss: {avg_g_loss:.4f}, D Loss: {avg_d_loss:.4f}, "
                    f"Content Loss: {avg_g_content_loss:.4f}, Elevation Loss: {avg_g_elevation_loss:.4f}, "
                    f"Intensity Loss: {avg_g_intensity_loss:.4f}, "
                    f"Val Loss: {avg_val_loss:.4f}, Val RMSE: {avg_val_rmse:.4f}")
        
        # Update learning rate schedulers
        gan_model.g_scheduler.step(avg_val_loss)
        gan_model.d_scheduler.step(avg_val_loss)
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_generator = copy.deepcopy(gan_model.generator)
            best_epoch = epoch
            
            # Save the best model so far
            torch.save({
                'epoch': epoch,
                'generator_state_dict': gan_model.generator.state_dict(),
                'discriminator_state_dict': gan_model.discriminator.state_dict(),
                'g_optimizer_state_dict': gan_model.g_optimizer.state_dict(),
                'd_optimizer_state_dict': gan_model.d_optimizer.state_dict(),
                'config': config,
                'val_loss': avg_val_loss,
            }, os.path.join(model_dir, 'best_model.pth'))
            
            logger.info(f"에포크 {epoch+1}에서 검증 손실 {avg_val_loss:.4f}로 새로운 최고 모델을 저장했습니다.")
        
        # Check for early stopping
        if early_stopping(avg_val_loss):
            logger.info(f"에포크 {epoch+1}에서 조기 중단이 트리거됨")
            break
        
        # Clean up GPU memory
        torch.cuda.empty_cache()
        gc.collect()
    
    writer.close()
    
    logger.info(f"훈련 완료. 최고 모델은 에포크 {best_epoch+1}에서 검증 손실 {best_val_loss:.4f}로 나타났습니다.")
    
    return best_generator, best_val_loss

# Testing function with metric evaluation
def test_gan_model_with_metrics(generator, test_data, threshold=1):
    test_loader = DataLoader(test_data, batch_size=1, shuffle=False)
    generator.eval()
    predictions = []
    ground_truths = []
    
    with torch.no_grad():
        for low_res, high_res in tqdm(test_loader, desc="테스트 중"):
            low_res = low_res.cuda()
            output = generator(low_res, (61, 101))
            pred = output.cpu().squeeze().numpy()
            
            if threshold is not None:
                pred = np.where(pred < threshold, 0, pred)
            
            predictions.append(pred)
            ground_truths.append(high_res.squeeze().numpy())  # Collect the actual high_res data
            
    # Compute evaluation metrics
    rmse, mae, mape, cc = evaluate_metrics(predictions, ground_truths)
    
    logger.info(f"테스트 결과 - RMSE: {rmse:.4f}, MAE: {mae:.4f}, MAPE: {mape:.4f}%, 상관 계수(CC): {cc:.4f}")
    
    return predictions, rmse, mae, mape, cc

# Function to compute RMSE, MAE, MAPE, CC between predictions and ground truth
def evaluate_metrics(predictions, ground_truths):
    rmse_list = []
    mae_list = []
    mape_list = []
    cc_list = []
    
    for pred, truth in zip(predictions, ground_truths):
        # Flatten the arrays for easier comparison
        pred_flat = pred.flatten()
        truth_flat = truth.flatten()
        
        # RMSE
        rmse = np.sqrt(np.mean((pred_flat - truth_flat) ** 2))
        rmse_list.append(rmse)
        
        # MAE
        mae = np.mean(np.abs(pred_flat - truth_flat))
        mae_list.append(mae)
        
        # MAPE (Note: To avoid division by zero, we mask small values in the denominator)
        mape = np.mean(np.abs((truth_flat - pred_flat) / (truth_flat + 1e-8))) * 100
        mape_list.append(mape)
        
        # Correlation Coefficient (CC)
        pred_mean = np.mean(pred_flat)
        truth_mean = np.mean(truth_flat)
        numerator = np.sum((pred_flat - pred_mean) * (truth_flat - truth_mean))
        denominator = np.sqrt(np.sum((pred_flat - pred_mean) ** 2) * np.sum((truth_flat - truth_mean) ** 2))
        cc = numerator / (denominator + 1e-8)  # To avoid division by zero
        cc_list.append(cc)
    
    # Average over all samples
    avg_rmse = np.mean(rmse_list)
    avg_mae = np.mean(mae_list)
    avg_mape = np.mean(mape_list)
    avg_cc = np.mean(cc_list)
    
    return avg_rmse, avg_mae, avg_mape, avg_cc

# Function to save predictions
def save_predictions_to_nc(predictions, dates, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    jeju_lat_start, jeju_lat_end = 33.1, 33.6
    jeju_lon_start, jeju_lon_end = 126.1, 127.1
    
    lat_vals = np.linspace(jeju_lat_start, jeju_lat_end, predictions[0].shape[0])
    lon_vals = np.linspace(jeju_lon_start, jeju_lon_end, predictions[0].shape[1])
    
    for pred, date in zip(predictions, dates):
        filename = os.path.join(output_folder, f"{date.strftime('%Y%m%d')}_GAN_Prediction.nc")
        
        try:
            with ncDataset(filename, 'w', format='NETCDF4') as nc:
                nc.createDimension('latitude', pred.shape[0])
                nc.createDimension('longitude', pred.shape[1])
                
                lats = nc.createVariable('latitude', 'f4', ('latitude',))
                lons = nc.createVariable('longitude', 'f4', ('longitude',))
                precip = nc.createVariable('precipitation', 'f4', ('latitude', 'longitude',))
                
                lats[:] = lat_vals
                lons[:] = lon_vals
                precip[:, :] = pred
                
                nc.description = 'GAN Downscaled Precipitation Prediction with DEM and Seasonal Encoding'
                nc.date = date.strftime('%Y-%m-%d')
            
            logger.info(f"{date.strftime('%Y-%m-%d')} 날짜에 대한 예측이 성공적으로 저장되었습니다.")
        except Exception as e:
            logger.error(f"{date.strftime('%Y-%m-%d')} 날짜에 대한 예측 저장 중 오류 발생: {str(e)}")

# Hyperparameter optimization wrapper
def optimize_hyperparameters(train_data, val_data, dem_data, n_trials=30, study_name="rainfall_gan_optimization"):
    logger.info("하이퍼파라미터 최적화 시작")
    
    # Create Optuna study
    study = optuna.create_study(
        direction="minimize",
        study_name=study_name,
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)
    )
    
    # Run optimization
    study.optimize(
        lambda trial: objective(trial, train_data, val_data, dem_data), 
        n_trials=n_trials
    )
    
    # Get best parameters
    best_params = study.best_params
    best_value = study.best_value
    
    logger.info(f"최적화 완료. 최고 검증 손실: {best_value:.4f}")
    logger.info(f"최적 파라미터: {best_params}")
    
    # Save optimization results
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
    
    with open(os.path.join(log_dir, 'best_hyperparameters.json'), 'w') as f:
        json.dump(best_params, f, indent=4)
    
    # Add some fixed parameters to the configuration
    config = best_params.copy()
    config['early_stopping_patience'] = 15  # Add early stopping patience
    
    return config

# Main function
def main():
    start_train = datetime.date(2004, 1, 1)
    end_train = datetime.date(2018, 12, 31)
    start_val = datetime.date(2019, 1, 1)
    end_val = datetime.date(2021, 12, 31)
    start_test = datetime.date(2022, 1, 1)
    end_test = datetime.date(2023, 12, 31)

    low_res_folder = r'C:\Yeonji\GPM\Input'
    high_res_folder = r'C:\Yeonji\2025.01.Drought\2004\1.CoKriging(Daily)'
    dem_file = r'C:\Yeonji\2025.01.Drought\DEM(merge).tif'
    output_folder = r'C:\Yeonji\2025.01.Drought/2004/GAN_Output(050615)'
    model_dir = r'C:\Yeonji\2025.01.Drought/2004/GAN_Output(050615)(Models)'

    # Set up output directories
    for directory in [output_folder, model_dir, log_dir]:
        if not os.path.exists(directory):
            os.makedirs(directory)

    logger.info("RainfallGAN 훈련 프로세스 시작")
    
    # 우선 디렉토리가 존재하는지 확인
    for directory, name in [(low_res_folder, "저해상도 폴더"), 
                           (high_res_folder, "고해상도 폴더"), 
                           (dem_file, "DEM 파일")]:
        if os.path.exists(directory):
            if os.path.isfile(directory):
                logger.info(f"{name}이(가) 파일로 존재합니다")
            else:
                try:
                    files = os.listdir(directory)
                    logger.info(f"{name}이(가) {len(files)}개의 파일과 함께 존재합니다. 샘플: {files[:5] if files else '파일 없음'}")
                except Exception as e:
                    logger.error(f"{name} 액세스 중 오류 발생: {str(e)}")
        else:
            logger.error(f"{name}이(가) 존재하지 않습니다: {directory}")
    
    # 데이터셋 생성 전에 파일 패턴 확인 (새로 추가된 부분)
    low_res_pattern, high_res_pattern = check_file_patterns(low_res_folder, high_res_folder)
    
    logger.info("데이터셋을 생성하는 중...")
    
    train_data = RainfallDataset(low_res_folder, high_res_folder, dem_file, start_train, end_train)
    val_data = RainfallDataset(low_res_folder, high_res_folder, dem_file, start_val, end_val)
    test_data = RainfallDataset(low_res_folder, high_res_folder, dem_file, start_test, end_test)

    logger.info(f"훈련 샘플: {len(train_data)}")
    logger.info(f"검증 샘플: {len(val_data)}")
    logger.info(f"테스트 샘플: {len(test_data)}")

    # 유효한 데이터셋이 있는지 확인
    if len(train_data) == 0 or len(val_data) == 0 or len(test_data) == 0:
        logger.error("하나 이상의 데이터셋에 유효한 데이터 포인트가 없습니다. 경로와 파일 이름 패턴을 확인하세요.")
        return  # 실행 중단

    if torch.cuda.is_available():
        logger.info(f"GPU 사용 가능: {torch.cuda.get_device_name(0)}")
    else:
        logger.warning("GPU를 사용할 수 없어 CPU를 사용합니다. 이는 매우 느릴 수 있습니다!")

    try:
        # 첫 번째로 하이퍼파라미터 최적화 수행
        logger.info("하이퍼파라미터 최적화 시작...")
        config = optimize_hyperparameters(train_data, val_data, train_data.dem_data, n_trials=30)
        
        # 최적화된 하이퍼파라미터로 모델 훈련
        logger.info("최적화된 하이퍼파라미터로 GAN 모델 훈련 중...")
        best_generator, best_val_loss = train_gan_model(
            train_data, 
            val_data, 
            train_data.dem_data, 
            config=config, 
            epochs=150,
            model_dir=model_dir
        )

        # 모델 테스트 및 메트릭 평가
        logger.info("메트릭과 함께 예측 생성 중...")
        predictions, rmse, mae, mape, cc = test_gan_model_with_metrics(
            best_generator, 
            test_data, 
            threshold=1
        )

        # 예측 저장
        logger.info("예측 저장 중...")
        save_predictions_to_nc(predictions, test_data.valid_data, output_folder)

        # 최종 메트릭 저장
        metrics = {
            'rmse': rmse,
            'mae': mae,
            'mape': mape,
            'cc': cc,
            'best_val_loss': best_val_loss
        }
        
        with open(os.path.join(model_dir, 'final_metrics.json'), 'w') as f:
            json.dump(metrics, f, indent=4)

        logger.info("훈련 및 평가 완료!")
        logger.info(f"최종 RMSE: {rmse:.4f}, MAE: {mae:.4f}, MAPE: {mape:.4f}%, CC: {cc:.4f}")

    except Exception as e:
        logger.error(f"훈련 프로세스 중 오류 발생: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())

    finally:
        torch.cuda.empty_cache()
        gc.collect()

if __name__ == "__main__":
    main()

2025-05-06 15:37:32,375 - RainfallGAN - INFO - RainfallGAN 훈련 프로세스 시작
2025-05-06 15:37:32,379 - RainfallGAN - INFO - 저해상도 폴더이(가) 7304개의 파일과 함께 존재합니다. 샘플: ['20040101.nc4', '20040102.nc4', '20040103.nc4', '20040104.nc4', '20040105.nc4']
2025-05-06 15:37:32,392 - RainfallGAN - INFO - 고해상도 폴더이(가) 7305개의 파일과 함께 존재합니다. 샘플: ['20040101_Cokriging_Precipitation.nc', '20040102_Cokriging_Precipitation.nc', '20040103_Cokriging_Precipitation.nc', '20040104_Cokriging_Precipitation.nc', '20040105_Cokriging_Precipitation.nc']
2025-05-06 15:37:32,393 - RainfallGAN - INFO - DEM 파일이(가) 파일로 존재합니다
2025-05-06 15:37:32,393 - RainfallGAN - INFO - 파일 이름 패턴 분석 중...
2025-05-06 15:37:32,398 - RainfallGAN - INFO - 저해상도 파일 예시: ['20040101.nc4', '20040102.nc4', '20040103.nc4', '20040104.nc4', '20040105.nc4', '20040106.nc4', '20040108.nc4', '20040109.nc4', '20040110.nc4', '20040111.nc4']
2025-05-06 15:37:32,408 - RainfallGAN - INFO - 고해상도 파일 예시: ['20040101_Cokriging_Precipitation.nc', '20040102_Cokriging_Precipitation.

In [7]:
import os
import glob
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

# 입력 폴더: .nc 파일들이 저장된 폴더
input_folder = r"C:\Yeonji\2025.01.Drought/2004/GAN_Output(050615)"

# 출력 폴더: PNG 파일들을 저장할 폴더
output_png_folder = r"C:\Yeonji\2025.01.Drought/2004/GAN_Output(050615)(png)"
if not os.path.exists(output_png_folder):
    os.makedirs(output_png_folder)

# 입력 폴더의 모든 .nc 파일 검색
nc_files = glob.glob(os.path.join(input_folder, "*_GAN_Prediction.nc"))

if not nc_files:
    print("입력 폴더에 GAN_Prediction.nc 파일이 없습니다.")
else:
    print(f"총 {len(nc_files)} 개의 NC 파일을 찾았습니다.")

# 각 파일별로 시각화
for nc_file in nc_files:
    try:
        # 파일 이름에서 날짜 추출 (YYYYMMDD)
        filename = os.path.basename(nc_file)
        date_str = filename.split('_')[0]  # YYYYMMDD 형식 추출
        
        # xarray로 데이터 로드
        ds = xr.open_dataset(nc_file)
        
        # precipitation 변수가 있는지 확인
        if "precipitation" in ds.data_vars:
            da = ds["precipitation"]
        else:
            # 첫 번째 변수 사용
            da = list(ds.data_vars.values())[0]
            print(f"'precipitation' 변수가 없어 {list(ds.data_vars.keys())[0]} 변수를 시각화합니다.")
        
        # 결측값 확인
        missing_pct = (np.isnan(da.values).sum() / da.size) * 100
        if missing_pct > 0:
            print(f"{filename}: 결측값 비율 {missing_pct:.2f}%")
        
        # 강수 데이터 범위를 고려한 색상 맵 설정
        plt.figure(figsize=(10, 8))
        
        # 0값이 많은 강수 데이터를 위한 설정
        vmax = np.nanpercentile(da.values, 99)  # 이상치 제외
        
        # 강수량에 적합한 색상맵 사용 (Blues는 강수에 적합)
        im = da.plot(cmap="Blues", vmin=0, vmax=vmax, add_colorbar=True)
        plt.colorbar(im, label="Precipitation (mm)")
        
        # 제목 및 축 레이블 설정
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}"
        plt.title(f"제주도 상세 강수량 예측 (GAN)\n{formatted_date}", fontsize=14)
        plt.xlabel("경도 (Longitude)", fontsize=12)
        plt.ylabel("위도 (Latitude)", fontsize=12)
        
        # 격자선 추가
        plt.grid(alpha=0.3, linestyle='--')
        
        # 저장할 PNG 파일 경로
        png_file = os.path.join(output_png_folder, os.path.splitext(filename)[0] + ".png")
        plt.savefig(png_file, dpi=200, bbox_inches="tight")
        plt.close()
        print(f"PNG 저장 완료: {png_file}")
        
    except Exception as e:
        print(f"{nc_file} 처리 중 오류 발생: {e}")

print("시각화 작업 완료!")

총 730 개의 NC 파일을 찾았습니다.


C:\Users\정연지\AppData\Local\Temp\ipykernel_20280\2524271508.py:67: UserWarning: Glyph 50948 (\N{HANGUL SYLLABLE WI}) missing from font(s) DejaVu Sans.
  plt.savefig(png_file, dpi=200, bbox_inches="tight")
C:\Users\정연지\AppData\Local\Temp\ipykernel_20280\2524271508.py:67: UserWarning: Glyph 46020 (\N{HANGUL SYLLABLE DO}) missing from font(s) DejaVu Sans.
  plt.savefig(png_file, dpi=200, bbox_inches="tight")
C:\Users\정연지\AppData\Local\Temp\ipykernel_20280\2524271508.py:67: UserWarning: Glyph 51228 (\N{HANGUL SYLLABLE JE}) missing from font(s) DejaVu Sans.
  plt.savefig(png_file, dpi=200, bbox_inches="tight")
C:\Users\정연지\AppData\Local\Temp\ipykernel_20280\2524271508.py:67: UserWarning: Glyph 51452 (\N{HANGUL SYLLABLE JU}) missing from font(s) DejaVu Sans.
  plt.savefig(png_file, dpi=200, bbox_inches="tight")
C:\Users\정연지\AppData\Local\Temp\ipykernel_20280\2524271508.py:67: UserWarning: Glyph 49345 (\N{HANGUL SYLLABLE SANG}) missing from font(s) DejaVu Sans.
  plt.savefig(png_file, dpi=200,

PNG 저장 완료: C:\Yeonji\2025.01.Drought/2004/GAN_Output(050615)(png)\20220101_GAN_Prediction.png
PNG 저장 완료: C:\Yeonji\2025.01.Drought/2004/GAN_Output(050615)(png)\20220102_GAN_Prediction.png
PNG 저장 완료: C:\Yeonji\2025.01.Drought/2004/GAN_Output(050615)(png)\20220103_GAN_Prediction.png
PNG 저장 완료: C:\Yeonji\2025.01.Drought/2004/GAN_Output(050615)(png)\20220104_GAN_Prediction.png
PNG 저장 완료: C:\Yeonji\2025.01.Drought/2004/GAN_Output(050615)(png)\20220105_GAN_Prediction.png
PNG 저장 완료: C:\Yeonji\2025.01.Drought/2004/GAN_Output(050615)(png)\20220106_GAN_Prediction.png
PNG 저장 완료: C:\Yeonji\2025.01.Drought/2004/GAN_Output(050615)(png)\20220107_GAN_Prediction.png
PNG 저장 완료: C:\Yeonji\2025.01.Drought/2004/GAN_Output(050615)(png)\20220108_GAN_Prediction.png
PNG 저장 완료: C:\Yeonji\2025.01.Drought/2004/GAN_Output(050615)(png)\20220109_GAN_Prediction.png
PNG 저장 완료: C:\Yeonji\2025.01.Drought/2004/GAN_Output(050615)(png)\20220110_GAN_Prediction.png
PNG 저장 완료: C:\Yeonji\2025.01.Drought/2004/GAN_Output(050615)

In [ ]:
import os
import datetime
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from netCDF4 import Dataset as ncDataset
import rasterio
from tqdm import tqdm
import logging
import json

In [ ]:
# 로깅 설정
log_dir = "rainfall_inference_logs"
if not os.path.exists(log_dir):
    os.makedirs(log_dir)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(log_dir, 'inference.log')),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger('RainfallGAN_Inference')

# Generator 모델 구조 재정의
class Generator(nn.Module):
    def __init__(self, dropout_rate=0.3, attention_channels=256):
        super(Generator, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(4, 64, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, attention_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(attention_channels),
            nn.LeakyReLU(0.2)
        )
        
        # Enhanced attention module
        self.attention = nn.Sequential(
            nn.Conv2d(attention_channels, attention_channels, kernel_size=1),
            nn.BatchNorm2d(attention_channels),
            nn.ReLU(),
            nn.Conv2d(attention_channels, attention_channels, kernel_size=1),
            nn.Sigmoid()
        )
        
        # Rainfall amplification branch
        self.rainfall_amp = nn.Sequential(
            nn.Conv2d(attention_channels, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 1, kernel_size=1),
            nn.ReLU()
        )
        
        self.decoder = nn.Sequential(
            nn.Conv2d(attention_channels, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Dropout2d(dropout_rate),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Dropout2d(dropout_rate),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            nn.Conv2d(32, 1, kernel_size=3, padding=1),
            nn.ReLU()
        )
        
        # Enhanced rainfall intensity layer with skip connections
        self.enhancement = nn.ModuleList([
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.Conv2d(32, 1, kernel_size=1)
        ])
        
    def forward(self, x, target_shape):
        # Encoder
        enc = self.encoder(x)
        
        # Enhanced attention
        att = self.attention(enc)
        enc_att = enc * att
        
        # Rainfall amplification branch
        amp = self.rainfall_amp(enc_att)
        
        # Decoder
        dec = self.decoder(enc_att)
        
        # Enhancement with skip connections
        enhanced = dec
        for i, layer in enumerate(self.enhancement):
            if i == len(self.enhancement) - 1:
                enhanced = layer(enhanced) + amp  # Add amplification
            else:
                enhanced = F.relu(layer(enhanced))
        
        # Ensure non-negative values and apply slight amplification
        enhanced = F.relu(enhanced) * 1.2  # Amplification factor
        
        # Interpolate to target shape
        output = torch.nn.functional.interpolate(
            enhanced, 
            size=target_shape, 
            mode='bilinear', 
            align_corners=False
        )
        
        return output

# 데이터 처리 함수들
def normalize_data(data):
    mean = np.mean(data)
    std = np.std(data)
    if std == 0:
        return data - mean
    return (data - mean) / (std + 1e-8)

def load_nc_file(file_path, var_name):
    try:
        with ncDataset(file_path, 'r') as nc_file:
            data = nc_file.variables[var_name][:].astype(np.float32)
            return data
    except FileNotFoundError:
        logger.warning(f"파일 {file_path}을(를) 찾을 수 없습니다.")
        return None
    except Exception as e:
        logger.error(f"{file_path} 로드 중 오류 발생: {str(e)}")
        return None

def load_and_resize_nc_file(file_path, var_name, target_shape):
    try:
        with ncDataset(file_path, 'r') as nc_file:
            data = nc_file.variables[var_name][:].astype(np.float32)
            if len(data.shape) == 3:
                data = data.squeeze(0)
            data = torch.tensor(data, dtype=torch.float32)
            data = torch.nn.functional.interpolate(
                data.unsqueeze(0).unsqueeze(0), 
                size=target_shape, 
                mode='bilinear', 
                align_corners=False
            ).squeeze()
            return data.numpy()
    except FileNotFoundError:
        logger.warning(f"파일 {file_path}을(를) 찾을 수 없습니다.")
        return None
    except Exception as e:
        logger.error(f"{file_path} 리사이징 중 오류 발생: {str(e)}")
        return None

def load_dem_data(dem_file, target_shape=(61, 101)):
    try:
        with rasterio.open(dem_file) as dem_src:
            dem_data = dem_src.read(1)
            dem_data = normalize_data(dem_data)
            dem_data = torch.tensor(dem_data, dtype=torch.float32)
            dem_data = torch.nn.functional.interpolate(
                dem_data.unsqueeze(0).unsqueeze(0), 
                size=target_shape, 
                mode='bilinear', 
                align_corners=False
            ).squeeze()
            return dem_data.numpy()
    except FileNotFoundError:
        logger.error(f"DEM 파일 {dem_file}을(를) 찾을 수 없습니다.")
        return None
    except Exception as e:
        logger.error(f"DEM 파일 {dem_file} 로드 중 오류 발생: {str(e)}")
        return None

def save_prediction_to_nc(prediction, date, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # 제주도 좌표 경계 설정 (원본 코드와 일치)
    jeju_lat_start, jeju_lat_end = 33.1, 33.6
    jeju_lon_start, jeju_lon_end = 126.1, 127.1
    
    lat_vals = np.linspace(jeju_lat_start, jeju_lat_end, prediction.shape[0])
    lon_vals = np.linspace(jeju_lon_start, jeju_lon_end, prediction.shape[1])
    
    filename = os.path.join(output_folder, f"{date.strftime('%Y%m%d')}_GAN_Prediction.nc")
    
    try:
        with ncDataset(filename, 'w', format='NETCDF4') as nc:
            nc.createDimension('latitude', prediction.shape[0])
            nc.createDimension('longitude', prediction.shape[1])
            
            lats = nc.createVariable('latitude', 'f4', ('latitude',))
            lons = nc.createVariable('longitude', 'f4', ('longitude',))
            precip = nc.createVariable('precipitation', 'f4', ('latitude', 'longitude',))
            
            lats[:] = lat_vals
            lons[:] = lon_vals
            precip[:, :] = prediction
            
            nc.description = 'GAN Downscaled Precipitation Prediction with DEM and Seasonal Encoding'
            nc.date = date.strftime('%Y-%m-%d')
        
        logger.info(f"{date.strftime('%Y-%m-%d')} 날짜에 대한 예측이 성공적으로 저장되었습니다.")
        return True
    except Exception as e:
        logger.error(f"{date.strftime('%Y-%m-%d')} 날짜에 대한 예측 저장 중 오류 발생: {str(e)}")
        return False

# 메인 추론 함수
def inference_all_data(low_res_folder, dem_file, model_path, output_folder, start_date, end_date, config=None):
    """
    전체 기간에 대해 GAN 모델을 사용하여 저해상도 데이터를 고해상도로 변환
    
    Args:
        low_res_folder: 저해상도 데이터 폴더 경로
        dem_file: DEM 파일 경로
        model_path: 훈련된 모델 파일 경로
        output_folder: 결과 저장 폴더 경로
        start_date: 시작 날짜 (datetime.date)
        end_date: 종료 날짜 (datetime.date)
        config: 모델 구성 파라미터 (None인 경우 모델 파일에서 로드)
    """
    # 결과 폴더 생성
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        
    # CUDA 사용 가능 여부 확인
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"사용 장치: {device}")
    
    # 모델 및 설정 로드
    try:
        checkpoint = torch.load(model_path, map_location=device)
        
        if config is None:
            if 'config' in checkpoint:
                config = checkpoint['config']
                logger.info(f"모델 파일에서 설정 로드: {config}")
            else:
                # 기본 설정 값 사용
                config = {
                    'generator_dropout': 0.3,
                    'attention_channels': 256,
                }
                logger.warning(f"모델 파일에서 설정을 찾을 수 없어 기본 설정 사용: {config}")
        
        # Generator 모델 초기화 및 가중치 로드
        generator = Generator(
            dropout_rate=config.get('generator_dropout', 0.3),
            attention_channels=config.get('attention_channels', 256)
        ).to(device)
        
        generator.load_state_dict(checkpoint['generator_state_dict'])
        generator.eval()
        logger.info("Generator 모델 로드 완료")
        
    except Exception as e:
        logger.error(f"모델 로드 중 오류 발생: {str(e)}")
        return
    
    # DEM 데이터 로드
    dem_data = load_dem_data(dem_file)
    if dem_data is None:
        logger.error("DEM 데이터를 로드할 수 없어 종료합니다.")
        return
    
    # 전체 날짜 목록 생성
    dates = [start_date + datetime.timedelta(days=i) for i in range((end_date - start_date).days + 1)]
    logger.info(f"총 {len(dates)}일 처리 예정 ({start_date} ~ {end_date})")
    
    # 성공 및 실패 카운트
    success_count = 0
    failed_count = 0
    skipped_count = 0
    
    # 각 날짜별로 처리
    for date in tqdm(dates, desc="날짜별 다운스케일링 진행 중"):
        try:
            # 날짜별 저해상도 파일 경로
            low_res_file = os.path.join(low_res_folder, date.strftime("%Y%m%d") + ".nc4")
            output_file = os.path.join(output_folder, f"{date.strftime('%Y%m%d')}_GAN_Prediction.nc")
            
            # 이미 처리된 파일 건너뛰기 (필요시 주석 해제)
            if os.path.exists(output_file):
                logger.info(f"{date.strftime('%Y-%m-%d')} 날짜는 이미 처리되었습니다. 건너뜁니다.")
                skipped_count += 1
                continue
            
            # 저해상도 파일 존재 여부 확인
            if not os.path.exists(low_res_file):
                logger.warning(f"{date.strftime('%Y-%m-%d')} 날짜의 저해상도 파일이 없습니다. 건너뜁니다.")
                failed_count += 1
                continue
            
            # 저해상도 데이터 로드 및 전처리
            low_res_data = load_and_resize_nc_file(low_res_file, 'precipitation', target_shape=(61, 101))
            
            if low_res_data is None:
                logger.warning(f"{date.strftime('%Y-%m-%d')} 날짜의 저해상도 데이터를 로드할 수 없습니다. 건너뜁니다.")
                failed_count += 1
                continue
            
            # 데이터 정규화
            low_res_data = normalize_data(low_res_data)
            
            # 월별 정보 추가 (사인, 코사인)
            month = date.month
            month_sin = np.sin(2 * np.pi * month / 12)
            month_cos = np.cos(2 * np.pi * month / 12)
            
            month_sin_map = np.full(low_res_data.shape, month_sin, dtype=np.float32)
            month_cos_map = np.full(low_res_data.shape, month_cos, dtype=np.float32)
            
            # 텐서 변환 및 채널 결합
            low_res_data = torch.tensor(low_res_data, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            dem_tensor = torch.tensor(dem_data, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            month_sin_map = torch.tensor(month_sin_map, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            month_cos_map = torch.tensor(month_cos_map, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            
            # 입력 데이터 결합: [저해상도, dem, sin(month), cos(month)]
            input_data = torch.cat([low_res_data, dem_tensor, month_sin_map, month_cos_map], dim=1).to(device)
            
            # 추론 수행
            with torch.no_grad():
                output = generator(input_data, (61, 101))
                prediction = output.cpu().squeeze().numpy()
                
                # 음수값 제거
                prediction = np.maximum(prediction, 0)
            
            # 예측 결과 저장
            if save_prediction_to_nc(prediction, date, output_folder):
                success_count += 1
            else:
                failed_count += 1
                
        except Exception as e:
            logger.error(f"{date.strftime('%Y-%m-%d')} 날짜 처리 중 오류 발생: {str(e)}")
            failed_count += 1
    
    # 최종 결과 요약
    logger.info(f"처리 완료: 성공 {success_count}건, 실패 {failed_count}건, 건너뜀 {skipped_count}건")
    return success_count, failed_count, skipped_count

# 메인 실행 함수
def main():
    # 경로 설정
    low_res_folder = r'C:\Yeonji\GPM\Input'
    dem_file = r'C:\Yeonji\2025.01.Drought\DEM(merge).tif'
    model_path = r'C:\Yeonji\2025.01.Drought/2004/GAN_Output(050615)(Models)/best_model.pth'
    output_folder = r'C:\Yeonji\2025.01.Drought/2004/Full_Period_GAN_Output(050615)'
    
    # 날짜 범위 설정
    start_date = datetime.date(2004, 1, 1)
    end_date = datetime.date(2023, 12, 31)
    
    # 시작 시간 기록
    start_time = datetime.datetime.now()
    logger.info(f"시작 시간: {start_time}")
    
    # 추론 실행
    success_count, failed_count, skipped_count = inference_all_data(
        low_res_folder=low_res_folder,
        dem_file=dem_file,
        model_path=model_path,
        output_folder=output_folder,
        start_date=start_date,
        end_date=end_date
    )
    
    # 종료 시간 기록 및 소요 시간 계산
    end_time = datetime.datetime.now()
    elapsed_time = end_time - start_time
    
    logger.info(f"종료 시간: {end_time}")
    logger.info(f"총 소요 시간: {elapsed_time}")
    logger.info(f"처리 결과: 총 {success_count + failed_count + skipped_count}건 중 "
                f"성공 {success_count}건, 실패 {failed_count}건, 건너뜀 {skipped_count}건")

if __name__ == "__main__":
    main()

## 뭐 임마!

In [ ]:
# ——————————————————————————
# 1) 설정부
# ——————————————————————————
import os
import re
import glob
import numpy as np
import xarray as xr
import rioxarray
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# 마스크 래스터 (.tif) 경로
mask_path   = r"C:/Yeonji/2025.01.Drought/0.Rawdata/Jeju(Raster-0.1).tif"

# 입력·출력 폴더
input_dir   = r"C:\Yeonji\GPM\Input"
output_dir  = r"C:\Yeonji\GPM\Input(Clip)"
os.makedirs(output_dir, exist_ok=True)

# 흰색→진파랑 그라데이션
cmap = LinearSegmentedColormap.from_list("white_blue", ["white", "blue"])

# ——————————————————————————
# 2) 마스크 불러오기
# ——————————————————————————
mask = rioxarray.open_rasterio(mask_path)[0]
if "band" in mask.dims:
    mask = mask.drop_vars("band")
mask = mask.rio.write_crs("EPSG:4326", inplace=True)

# ——————————————————————————
# 3) NC4 순회 및 처리
# ——————————————————————————
nc_files = glob.glob(os.path.join(input_dir, "*.nc4"))
if not nc_files:
    raise FileNotFoundError(f".nc4 파일을 찾지 못했어요: {input_dir}")

for fp in nc_files:
    ds = xr.open_dataset(fp)
    da = ds["precipitation"]
    if "time" in da.dims:
        da = da.isel(time=0)
    da = da.rio.write_crs("EPSG:4326")

    # 마스크 재투영
    if mask.rio.crs != da.rio.crs:
        mask_proj = mask.rio.reproject(da.rio.crs)
    else:
        mask_proj = mask
    mask_match = mask_proj.rio.reproject_match(da)

    # 클리핑
    da_clip = da.where(~np.isnan(mask_match), drop=False)

    # 2D 로 축소
    da2 = da_clip.squeeze()
    extras = [d for d in da2.dims if d not in ("lat", "lon")]
    if extras:
        da2 = da2.isel({d: 0 for d in extras})

    # 날짜 정보
    m = re.search(r"\d{4}\.\d{2}", fp)
    date_info = m.group(0) if m else os.path.splitext(os.path.basename(fp))[0]

    # 디버깅 출력
    print(f"데이터 배열 모양: {da2.values.shape}")
    print(f"Lon 차원: {da2.lon.values.shape}")
    print(f"Lat 차원: {da2.lat.values.shape}")
    
    # 플롯 - imshow 방식으로 변경 (pcolormesh 대신)
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # 데이터 범위 계산
    lon_min, lon_max = da2.lon.min().item(), da2.lon.max().item()
    lat_min, lat_max = da2.lat.min().item(), da2.lat.max().item()
    
    # imshow로 표시 (데이터 배열 방향 전환 필요할 수 있음)
    im = ax.imshow(da2.values, 
                  extent=[lon_min, lon_max, lat_min, lat_max],
                  origin='lower',  # 위에서 아래로 표시
                  cmap=cmap,
                  aspect='auto')    # 지리 데이터이므로 자동 비율 적용
    
    ax.set_title(f"OBS_CoKriging {date_info}")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    fig.colorbar(im, ax=ax, label="Precip (mm)")

    # 저장
    base = os.path.splitext(os.path.basename(fp))[0]
    fig.savefig(os.path.join(output_dir, f"{base}.png"), dpi=300, bbox_inches="tight")
    plt.close(fig)

    # NetCDF 로 기록
    da2.to_netcdf(os.path.join(output_dir, os.path.basename(fp)), mode="w")
    ds.close()

print("✅ 작업 완료했습니다!")

데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
Lat 차원: (6,)
데이터 배열 모양: (10, 6)
Lon 차원: (10,)
L

MemoryError: Unable to allocate 26.2 MiB for an array with shape (1848, 1860) and data type int64

Error in callback <function _draw_all_if_interactive at 0x000001A0217FBAF0> (for post_execute):
